# ST-A² Ablation: Area Attention vs Baseline

**Spatiotemporal Area Attention for V-JEPA 2**

This notebook runs a synthetic-data ablation comparing:
- **Baseline**: Standard RoPE attention (full self-attention)
- **ST-A²**: RoPE Area Attention (partitioned spatiotemporal attention)

Both use the **real V-JEPA 2 model** (ViT-L encoder + predictor) with synthetic
random video tensors. No dataset download required.

**Config**: 256px, 16 frames, batch_size=1, ViT-L (24 layers).
Token grid: 16×16×8 = 2048 tokens → ~512 visible after masking.

**Metrics collected**: loss convergence, step time, peak memory, throughput.

**Hardware**: Colab T4 (16GB VRAM), FP16.

In [ ]:
# Cell 1: Setup & Install
import os
if not os.path.exists('vjepa2'):
    !git clone -b feat/st-a2-area-attention https://github.com/tarassh/vjepa2.git
else:
    # Pull latest changes if repo already cloned
    !cd vjepa2 && git pull origin feat/st-a2-area-attention
os.chdir('vjepa2')
!pip install -q timm
print('Setup complete.')

In [ ]:
# Cell 2: Imports & GPU Detection
import sys
import copy
import time
import gc

import torch
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

# Add repo root to path
if os.getcwd().endswith('vjepa2'):
    sys.path.insert(0, os.getcwd())
elif os.path.exists('vjepa2'):
    sys.path.insert(0, os.path.join(os.getcwd(), 'vjepa2'))

from app.vjepa.utils import init_video_model
from src.masks.multiseq_multiblock3d import _MaskGenerator
from src.masks.utils import apply_masks
from src.utils.logging import AverageMeter

# GPU detection
assert torch.cuda.is_available(), 'CUDA required'
device = torch.device('cuda')
gpu_name = torch.cuda.get_device_name(0)
props = torch.cuda.get_device_properties(0)
gpu_mem_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
print(f'GPU: {gpu_name} ({gpu_mem_gb:.1f} GB)')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')

In [ ]:
# Cell 3: Configuration
#
# Full V-JEPA 2 training resolution: 256px, 16 frames.
# batch_size=1 to fit on T4 16GB with ViT-L + activation checkpointing.
# Token grid: (256/16)^2 × (16/2) = 16×16×8 = 2048 tokens total.
# After ~75% masking: ~512 visible tokens — large enough for area attention gains.

SHARED = dict(
    model_name='vit_large',
    crop_size=256,         # Full resolution (16x16 spatial patches)
    patch_size=16,
    tubelet_size=2,
    num_frames=16,         # Full frame count (8 temporal tokens)
    batch_size=1,          # Minimal batch to fit T4 16GB
    pred_depth=12,
    pred_embed_dim=384,
    pred_num_heads=12,
    num_steps=150,
    warmup_steps=10,
    lr=5.25e-4,
    weight_decay=0.04,
    loss_exp=1.0,          # L1 loss
    ema_momentum=0.999,
)

CONFIGS = {
    'baseline': {
        **SHARED,
        'use_area_attention': False,
    },
    'st_a2': {
        **SHARED,
        'use_area_attention': True,
        'area_attention_layers': [0, 18],
        'area_spatial_splits': 2,
        'area_temporal_splits': 2,
        'area_residual_scale': 1.0,
    },
}

# T4 uses FP16 (no BF16 support)
DTYPE = torch.float16

# Mask config (matches V-JEPA 2 default: 8 small blocks + 2 large blocks)
MASK_CFGS = [
    dict(spatial_scale=(0.15, 0.15), temporal_scale=(1.0, 1.0),
         aspect_ratio=(0.75, 1.5), num_blocks=8, max_temporal_keep=1.0),
    dict(spatial_scale=(0.7, 0.7), temporal_scale=(1.0, 1.0),
         aspect_ratio=(0.75, 1.5), num_blocks=2, max_temporal_keep=1.0),
]

# Token count summary
H = W = SHARED['crop_size'] // SHARED['patch_size']  # 16
T = SHARED['num_frames'] // SHARED['tubelet_size']     # 8
total_tokens = H * W * T
print(f'Configs: {list(CONFIGS.keys())}')
print(f'Steps per config: {SHARED["num_steps"]}')
print(f'Resolution: {SHARED["crop_size"]}px, Frames: {SHARED["num_frames"]}, Batch: {SHARED["batch_size"]}')
print(f'Token grid: {H}x{W}x{T} = {total_tokens} total tokens')
print(f'After ~75% masking: ~{total_tokens // 4} visible tokens')
print(f'Area attention: 4 areas of ~{total_tokens // 16} visible tokens each')

In [ ]:
# Cell 4: Synthetic Data & Mask Generator
#
# Creates random video tensors and generates masks using
# V-JEPA 2's real _MaskGenerator (same as training).

def make_mask_generators(cfg):
    """Create mask generators matching training config."""
    generators = []
    for m in MASK_CFGS:
        gen = _MaskGenerator(
            crop_size=cfg['crop_size'],
            num_frames=cfg['num_frames'],
            spatial_patch_size=cfg['patch_size'],
            temporal_patch_size=cfg['tubelet_size'],
            spatial_pred_mask_scale=m['spatial_scale'],
            temporal_pred_mask_scale=m['temporal_scale'],
            aspect_ratio=m['aspect_ratio'],
            npred=m['num_blocks'],
            max_context_frames_ratio=m['max_temporal_keep'],
        )
        generators.append(gen)
    return generators


def make_synthetic_batch(cfg, mask_generators):
    """Generate one synthetic batch with masks.

    Returns:
        clips: list of [B, 3, T, H, W] tensors (one element for single fpc)
        masks_enc: list of lists of [B, K_enc] index tensors
        masks_pred: list of lists of [B, K_pred] index tensors
    """
    B = cfg['batch_size']
    T = cfg['num_frames']
    H = W = cfg['crop_size']

    # Random video tensor
    clip = torch.randn(B, 3, T, H, W, device=device)

    # Generate masks for each mask strategy
    all_masks_enc = []
    all_masks_pred = []
    for gen in mask_generators:
        masks_enc, masks_pred = gen(B)
        all_masks_enc.append(masks_enc.to(device))
        all_masks_pred.append(masks_pred.to(device))

    # Wrap in list (single fpc group)
    return [clip], [all_masks_enc], [all_masks_pred]


# Quick test
_gens = make_mask_generators(SHARED)
_clips, _me, _mp = make_synthetic_batch(SHARED, _gens)
print(f'Clip shape: {_clips[0].shape}')
print(f'Mask enc shapes: {[m.shape for m in _me[0]]}')
print(f'Mask pred shapes: {[m.shape for m in _mp[0]]}')
del _clips, _me, _mp, _gens
torch.cuda.empty_cache()

In [ ]:
# Cell 5: Model Builder
#
# Uses the real init_video_model() from V-JEPA 2.
# Creates encoder, predictor, target_encoder, optimizer.

def build_models(cfg):
    """Build encoder, predictor, target_encoder, optimizer, scaler."""
    num_mask_tokens = len(MASK_CFGS)  # one mask token per mask strategy

    encoder, predictor = init_video_model(
        device=device,
        patch_size=cfg['patch_size'],
        max_num_frames=cfg['num_frames'],
        tubelet_size=cfg['tubelet_size'],
        model_name=cfg['model_name'],
        crop_size=cfg['crop_size'],
        pred_depth=cfg['pred_depth'],
        pred_num_heads=cfg['pred_num_heads'],
        pred_embed_dim=cfg['pred_embed_dim'],
        uniform_power=True,
        use_mask_tokens=True,
        num_mask_tokens=num_mask_tokens,
        zero_init_mask_tokens=True,
        use_sdpa=True,
        use_rope=True,
        use_activation_checkpointing=True,  # save memory on T4
        use_area_attention=cfg['use_area_attention'],
        area_attention_layers=cfg.get('area_attention_layers'),
        area_spatial_splits=cfg.get('area_spatial_splits', 2),
        area_temporal_splits=cfg.get('area_temporal_splits', 2),
        area_residual_scale=cfg.get('area_residual_scale', 1.0),
    )

    target_encoder = copy.deepcopy(encoder)
    target_encoder.to(device)
    for p in target_encoder.parameters():
        p.requires_grad = False

    # Optimizer (simplified - no scheduler needed for short ablation)
    optimizer = torch.optim.AdamW(
        list(encoder.parameters()) + list(predictor.parameters()),
        lr=cfg['lr'],
        weight_decay=cfg['weight_decay'],
        betas=(0.9, 0.999),
    )
    scaler = torch.amp.GradScaler('cuda')

    # Count params
    enc_params = sum(p.numel() for p in encoder.parameters()) / 1e6
    pred_params = sum(p.numel() for p in predictor.parameters()) / 1e6
    print(f'  Encoder: {enc_params:.1f}M params')
    print(f'  Predictor: {pred_params:.1f}M params')

    return encoder, predictor, target_encoder, optimizer, scaler

print('build_models() defined.')

In [ ]:
# Cell 6: Training Step
#
# Mirrors app/vjepa/train.py lines 435-497:
# forward target (no_grad) -> forward context -> predictor -> L1 loss -> backward -> EMA

def train_step(encoder, predictor, target_encoder, optimizer, scaler,
               clips, masks_enc, masks_pred, loss_exp=1.0, momentum=0.999):
    """One V-JEPA 2 training step. Returns loss value."""

    def forward_target(c):
        with torch.no_grad():
            h = target_encoder(c)
            h = [F.layer_norm(hi, (hi.size(-1),)) for hi in h]
            return h

    def forward_context(c):
        z = encoder(c, masks_enc)
        z = predictor(z, masks_enc, masks_pred)
        return z

    def loss_fn(z, h):
        h = [apply_masks(hi, mi, concat=False) for hi, mi in zip(h, masks_pred)]
        loss, n = 0, 0
        for zi, hi in zip(z, h):
            for zij, hij in zip(zi, hi):
                loss += torch.mean(torch.abs(zij - hij) ** loss_exp) / loss_exp
                n += 1
        loss /= n
        return loss

    # Forward
    with torch.amp.autocast('cuda', dtype=DTYPE):
        h = forward_target(clips)
        z = forward_context(clips)
        loss = loss_fn(z, h)

    # Backward
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()

    # EMA update of target encoder
    with torch.no_grad():
        for param_q, param_k in zip(encoder.parameters(), target_encoder.parameters()):
            param_k.data.mul_(momentum).add_(param_q.data, alpha=1 - momentum)

    return float(loss)

print('train_step() defined.')

In [ ]:
# Cell 7: Run Ablation
#
# Runs num_steps training iterations, records metrics per step.

def run_ablation(name, cfg):
    """Run a single ablation config. Returns dict of metrics."""
    print(f'\n{"="*60}')
    print(f'Running: {name}')
    print(f'  Area attention: {cfg["use_area_attention"]}')
    print(f'{"="*60}')

    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    gc.collect()

    # Build models
    encoder, predictor, target_encoder, optimizer, scaler = build_models(cfg)
    mask_generators = make_mask_generators(cfg)

    num_steps = cfg['num_steps']
    losses = []
    step_times_ms = []

    # Warmup (3 steps, not recorded)
    print('  Warmup (3 steps)...')
    for _ in range(3):
        clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)
        _ = train_step(encoder, predictor, target_encoder, optimizer, scaler,
                       clips, masks_enc, masks_pred,
                       loss_exp=cfg['loss_exp'], momentum=cfg['ema_momentum'])
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()

    print(f'  Training ({num_steps} steps)...')
    for step in range(num_steps):
        clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)

        # Time the step with CUDA events
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)
        start_event.record()

        loss = train_step(encoder, predictor, target_encoder, optimizer, scaler,
                          clips, masks_enc, masks_pred,
                          loss_exp=cfg['loss_exp'], momentum=cfg['ema_momentum'])

        end_event.record()
        torch.cuda.synchronize()
        elapsed_ms = start_event.elapsed_time(end_event)

        losses.append(loss)
        step_times_ms.append(elapsed_ms)

        if (step + 1) % 25 == 0 or step == 0:
            avg_loss = np.mean(losses[-25:])
            avg_time = np.mean(step_times_ms[-25:])
            print(f'    Step {step+1:4d}/{num_steps}: loss={avg_loss:.4f}, time={avg_time:.1f}ms')

    peak_mem_mb = torch.cuda.max_memory_allocated() / 1024**2

    # Cleanup
    del encoder, predictor, target_encoder, optimizer, scaler
    torch.cuda.empty_cache()
    gc.collect()

    result = {
        'losses': losses,
        'step_times_ms': step_times_ms,
        'peak_mem_mb': peak_mem_mb,
        'avg_step_ms': np.mean(step_times_ms),
        'final_loss': np.mean(losses[-20:]),
        'throughput_steps_sec': 1000.0 / np.mean(step_times_ms),
    }
    print(f'  Done. Final loss={result["final_loss"]:.4f}, '
          f'avg step={result["avg_step_ms"]:.1f}ms, '
          f'peak mem={result["peak_mem_mb"]:.0f}MB')
    return result

print('run_ablation() defined.')

In [ ]:
# Cell 8: Execute Both Configs

results = {}
for name, cfg in CONFIGS.items():
    results[name] = run_ablation(name, cfg)

print('\n\nAll ablations complete!')

## Per-Layer Profiling

The cells below profile **where time is actually spent** inside the ViT-L encoder.
Each of the 24 Block layers has two components:
- **Attention** (norm1 → attn): QKV projection, RoPE, SDPA or area-attention, output projection
- **MLP/FFN** (norm2 → mlp): Two linear layers + activation

We hook into each Block to measure attention vs MLP time separately.
This reveals whether attention is actually the bottleneck at this token count.

In [ ]:
# Cell 9: Per-Layer Profiling — Instrument Block forward
#
# Monkey-patches Block.forward to time attention vs MLP separately.
# Runs 20 forward passes per config and averages the per-layer timings.

from src.models.utils.modules import Block, RoPEAttention, RoPEAreaAttention

def profile_encoder(cfg, num_runs=20):
    """Profile per-layer attention vs MLP time for one config."""
    print(f'\nProfiling: {"ST-A²" if cfg["use_area_attention"] else "Baseline"}')

    torch.cuda.empty_cache()
    gc.collect()

    encoder, predictor = init_video_model(
        device=device,
        patch_size=cfg['patch_size'],
        max_num_frames=cfg['num_frames'],
        tubelet_size=cfg['tubelet_size'],
        model_name=cfg['model_name'],
        crop_size=cfg['crop_size'],
        pred_depth=cfg['pred_depth'],
        pred_num_heads=cfg['pred_num_heads'],
        pred_embed_dim=cfg['pred_embed_dim'],
        uniform_power=True,
        use_mask_tokens=True,
        num_mask_tokens=len(MASK_CFGS),
        zero_init_mask_tokens=True,
        use_sdpa=True,
        use_rope=True,
        use_activation_checkpointing=False,  # Disable for accurate profiling
        use_area_attention=cfg['use_area_attention'],
        area_attention_layers=cfg.get('area_attention_layers'),
        area_spatial_splits=cfg.get('area_spatial_splits', 2),
        area_temporal_splits=cfg.get('area_temporal_splits', 2),
        area_residual_scale=cfg.get('area_residual_scale', 1.0),
    )
    encoder.eval()

    # Find all Block layers in the encoder backbone
    blocks = encoder.backbone.blocks
    num_layers = len(blocks)
    print(f'  Found {num_layers} Block layers')

    # Storage for timings: [layer_idx] -> {'attn': [], 'mlp': []}
    layer_timings = [{'attn': [], 'mlp': []} for _ in range(num_layers)]

    # Monkey-patch each Block's forward to record timings
    original_forwards = []
    for i, block in enumerate(blocks):
        original_forward = block.forward
        original_forwards.append(original_forward)

        def make_profiled_forward(block_ref, layer_idx):
            def profiled_forward(x, mask=None, attn_mask=None, T=None, H_patches=None, W_patches=None):
                # Time attention (norm1 + attn)
                torch.cuda.synchronize()
                t0 = time.perf_counter()
                if isinstance(block_ref.attn, (RoPEAttention, RoPEAreaAttention)):
                    y = block_ref.attn(block_ref.norm1(x), mask=mask, attn_mask=attn_mask,
                                       T=T, H_patches=H_patches, W_patches=W_patches)
                else:
                    y = block_ref.attn(block_ref.norm1(x), mask=mask, attn_mask=attn_mask)
                torch.cuda.synchronize()
                t1 = time.perf_counter()

                x_out = x + block_ref.drop_path(y)

                # Time MLP (norm2 + mlp)
                torch.cuda.synchronize()
                t2 = time.perf_counter()
                x_out = x_out + block_ref.drop_path(block_ref.mlp(block_ref.norm2(x_out)))
                torch.cuda.synchronize()
                t3 = time.perf_counter()

                layer_timings[layer_idx]['attn'].append((t1 - t0) * 1000)  # ms
                layer_timings[layer_idx]['mlp'].append((t3 - t2) * 1000)
                return x_out
            return profiled_forward

        block.forward = make_profiled_forward(block, i)

    mask_generators = make_mask_generators(cfg)

    # Warmup
    print('  Warmup (3 runs)...')
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
        for _ in range(3):
            clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)
            _ = encoder(clips, masks_enc)
    # Clear warmup timings
    for lt in layer_timings:
        lt['attn'].clear()
        lt['mlp'].clear()

    # Profile runs
    print(f'  Profiling ({num_runs} forward passes)...')
    with torch.no_grad(), torch.amp.autocast('cuda', dtype=DTYPE):
        for run in range(num_runs):
            clips, masks_enc, masks_pred = make_synthetic_batch(cfg, mask_generators)
            _ = encoder(clips, masks_enc)

    # Restore original forwards
    for i, block in enumerate(blocks):
        block.forward = original_forwards[i]

    # Aggregate results
    profile_data = []
    total_attn_ms = 0
    total_mlp_ms = 0
    for i in range(num_layers):
        attn_ms = np.mean(layer_timings[i]['attn'])
        mlp_ms = np.mean(layer_timings[i]['mlp'])
        attn_type = type(blocks[i].attn).__name__
        total_attn_ms += attn_ms
        total_mlp_ms += mlp_ms
        profile_data.append({
            'layer': i,
            'attn_type': attn_type,
            'attn_ms': attn_ms,
            'mlp_ms': mlp_ms,
            'total_ms': attn_ms + mlp_ms,
            'attn_pct': attn_ms / (attn_ms + mlp_ms) * 100,
        })

    # Cleanup
    del encoder, predictor
    torch.cuda.empty_cache()
    gc.collect()

    total_ms = total_attn_ms + total_mlp_ms
    print(f'  Total encoder forward: {total_ms:.1f}ms')
    print(f'    Attention: {total_attn_ms:.1f}ms ({total_attn_ms/total_ms*100:.1f}%)')
    print(f'    MLP/FFN:   {total_mlp_ms:.1f}ms ({total_mlp_ms/total_ms*100:.1f}%)')

    return profile_data

# Run profiling for both configs
profile_results = {}
for name, cfg in CONFIGS.items():
    profile_results[name] = profile_encoder(cfg)

print('\nProfiling complete!')

In [ ]:
# Cell 9: Summary Table

bl = results['baseline']
st = results['st_a2']

def delta_pct(base, new):
    if base == 0:
        return 0
    return (new - base) / abs(base) * 100

# Print config used
H = W = SHARED['crop_size'] // SHARED['patch_size']
T = SHARED['num_frames'] // SHARED['tubelet_size']
total_tokens = H * W * T

print('\n' + '='*70)
print('  ST-A\u00b2 ABLATION RESULTS')
print('='*70)
print()
print('  Config:')
print(f'    Model:       {SHARED["model_name"]}')
print(f'    Resolution:  {SHARED["crop_size"]}px, {SHARED["num_frames"]} frames')
print(f'    Batch size:  {SHARED["batch_size"]}')
print(f'    Patch size:  {SHARED["patch_size"]}, Tubelet: {SHARED["tubelet_size"]}')
print(f'    Token grid:  {H}x{W}x{T} = {total_tokens} total tokens')
print(f'    Visible:     ~{total_tokens // 4} tokens (after ~75% masking)')
print(f'    Dtype:       {DTYPE}')
print(f'    Steps:       {SHARED["num_steps"]}')
print(f'    GPU:         {gpu_name} ({gpu_mem_gb:.1f} GB)')
print()
print('  ST-A\u00b2 config:')
st_cfg = CONFIGS['st_a2']
print(f'    Area layers:     [{st_cfg["area_attention_layers"][0]}, {st_cfg["area_attention_layers"][1]}) of 24')
print(f'    Spatial splits:  {st_cfg["area_spatial_splits"]}')
print(f'    Temporal splits: {st_cfg["area_temporal_splits"]}')
print(f'    Num areas:       {st_cfg["area_spatial_splits"] * st_cfg["area_temporal_splits"]}')
print()
print(f'{"Metric":<30} {"Baseline":>12} {"ST-A\u00b2":>12} {"Delta":>12}')
print('-'*70)

rows = [
    ('Final Loss (last 20)',       bl['final_loss'],          st['final_loss'],          ''),
    ('Avg Step Time (ms)',         bl['avg_step_ms'],         st['avg_step_ms'],         ''),
    ('Peak Memory (MB)',           bl['peak_mem_mb'],         st['peak_mem_mb'],         ''),
    ('Throughput (steps/sec)',     bl['throughput_steps_sec'], st['throughput_steps_sec'], ''),
]

for label, v_bl, v_st, _ in rows:
    d = delta_pct(v_bl, v_st)
    sign = '+' if d >= 0 else ''
    print(f'{label:<30} {v_bl:>12.2f} {v_st:>12.2f} {sign}{d:>10.1f}%')

print('='*70)
print()

# Interpretation
mem_saving = delta_pct(bl['peak_mem_mb'], st['peak_mem_mb'])
speed_gain = delta_pct(bl['avg_step_ms'], st['avg_step_ms'])
loss_diff = delta_pct(bl['final_loss'], st['final_loss'])

print('Interpretation:')
if speed_gain < 0:
    print(f'  \u2705 ST-A\u00b2 is {abs(speed_gain):.1f}% FASTER per step')
else:
    print(f'  \u26a0\ufe0f  ST-A\u00b2 is {speed_gain:.1f}% slower per step')

if mem_saving < 0:
    print(f'  \u2705 ST-A\u00b2 uses {abs(mem_saving):.1f}% LESS peak memory')
else:
    print(f'  \u26a0\ufe0f  ST-A\u00b2 uses {mem_saving:.1f}% more peak memory')

if abs(loss_diff) < 5:
    print(f'  \u2705 Loss difference is small ({loss_diff:+.1f}%) - quality preserved')
else:
    print(f'  \u26a0\ufe0f  Loss difference is notable ({loss_diff:+.1f}%)')

In [ ]:
# Cell 11: Per-Layer Profiling Table

print('='*90)
print('  PER-LAYER PROFILING: Attention vs MLP Time (ms)')
print('='*90)
print()

for name, label in [('baseline', 'BASELINE (RoPEAttention)'), ('st_a2', 'ST-A² (RoPEAreaAttention layers 0-17)')]:
    data = profile_results[name]
    total_attn = sum(d['attn_ms'] for d in data)
    total_mlp = sum(d['mlp_ms'] for d in data)
    total = total_attn + total_mlp

    print(f'  {label}')
    print(f'  {"Layer":<8} {"Type":<22} {"Attn(ms)":>10} {"MLP(ms)":>10} {"Total(ms)":>10} {"Attn%":>8}')
    print(f'  {"-"*72}')
    for d in data:
        print(f'  {d["layer"]:<8} {d["attn_type"]:<22} {d["attn_ms"]:>10.2f} {d["mlp_ms"]:>10.2f} '
              f'{d["total_ms"]:>10.2f} {d["attn_pct"]:>7.1f}%')
    print(f'  {"-"*72}')
    print(f'  {"TOTAL":<8} {"":<22} {total_attn:>10.2f} {total_mlp:>10.2f} '
          f'{total:>10.2f} {total_attn/total*100:>7.1f}%')
    print()

# Compare attention time between configs
bl_attn = sum(d['attn_ms'] for d in profile_results['baseline'])
st_attn = sum(d['attn_ms'] for d in profile_results['st_a2'])
bl_mlp = sum(d['mlp_ms'] for d in profile_results['baseline'])
st_mlp = sum(d['mlp_ms'] for d in profile_results['st_a2'])
bl_total = bl_attn + bl_mlp
st_total = st_attn + st_mlp

print('='*70)
print('  COMPARISON SUMMARY')
print('='*70)
print(f'  {"Component":<20} {"Baseline(ms)":>14} {"ST-A²(ms)":>14} {"Delta":>10}')
print(f'  {"-"*60}')
d_attn = (st_attn - bl_attn) / bl_attn * 100
d_mlp = (st_mlp - bl_mlp) / bl_mlp * 100
d_total = (st_total - bl_total) / bl_total * 100
print(f'  {"Attention":<20} {bl_attn:>14.2f} {st_attn:>14.2f} {d_attn:>+9.1f}%')
print(f'  {"MLP/FFN":<20} {bl_mlp:>14.2f} {st_mlp:>14.2f} {d_mlp:>+9.1f}%')
print(f'  {"Total Encoder":<20} {bl_total:>14.2f} {st_total:>14.2f} {d_total:>+9.1f}%')
print(f'  {"Attn % of total":<20} {bl_attn/bl_total*100:>13.1f}% {st_attn/st_total*100:>13.1f}%')
print('='*70)

In [ ]:
# Cell 12: Per-Layer Profiling Charts

num_layers = len(profile_results['baseline'])
layers = np.arange(num_layers)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# --- Top Left: Stacked bar — Baseline ---
ax = axes[0, 0]
bl_data = profile_results['baseline']
attn_vals = [d['attn_ms'] for d in bl_data]
mlp_vals = [d['mlp_ms'] for d in bl_data]
ax.bar(layers, attn_vals, label='Attention', color='#2196F3', alpha=0.85)
ax.bar(layers, mlp_vals, bottom=attn_vals, label='MLP/FFN', color='#90CAF9', alpha=0.85)
ax.set_xlabel('Layer')
ax.set_ylabel('Time (ms)')
ax.set_title('Baseline: Per-Layer Time Breakdown')
ax.legend()
ax.set_xticks(layers[::2])

# --- Top Right: Stacked bar — ST-A² ---
ax = axes[0, 1]
st_data = profile_results['st_a2']
attn_vals_st = [d['attn_ms'] for d in st_data]
mlp_vals_st = [d['mlp_ms'] for d in st_data]
colors_attn = ['#FF5722' if d['attn_type'] == 'RoPEAreaAttention' else '#2196F3' for d in st_data]
for i in range(num_layers):
    ax.bar(i, attn_vals_st[i], color=colors_attn[i], alpha=0.85,
           label='Area Attention' if i == 0 else ('Full Attention' if i == 18 else ''))
    ax.bar(i, mlp_vals_st[i], bottom=attn_vals_st[i], color='#FFAB91', alpha=0.85,
           label='MLP/FFN' if i == 0 else '')
# Add divider line at layer 18
ax.axvline(x=17.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.text(8.5, ax.get_ylim()[1] * 0.95, 'Area Attn', ha='center', fontsize=9, color='#FF5722')
ax.text(20.5, ax.get_ylim()[1] * 0.95, 'Full', ha='center', fontsize=9, color='#2196F3')
ax.set_xlabel('Layer')
ax.set_ylabel('Time (ms)')
ax.set_title('ST-A²: Per-Layer Time Breakdown')
ax.legend(loc='upper left')
ax.set_xticks(layers[::2])

# --- Bottom Left: Attention time comparison per layer ---
ax = axes[1, 0]
bl_attn_vals = [d['attn_ms'] for d in profile_results['baseline']]
st_attn_vals = [d['attn_ms'] for d in profile_results['st_a2']]
width = 0.35
ax.bar(layers - width/2, bl_attn_vals, width, label='Baseline', color='#2196F3', alpha=0.85)
ax.bar(layers + width/2, st_attn_vals, width, label='ST-A²', color='#FF5722', alpha=0.85)
ax.axvline(x=17.5, color='gray', linestyle='--', linewidth=1, alpha=0.7)
ax.set_xlabel('Layer')
ax.set_ylabel('Attention Time (ms)')
ax.set_title('Attention Time: Baseline vs ST-A² (per layer)')
ax.legend()
ax.set_xticks(layers[::2])

# --- Bottom Right: Pie chart — time breakdown ---
ax = axes[1, 1]
bl_total_attn = sum(d['attn_ms'] for d in profile_results['baseline'])
bl_total_mlp = sum(d['mlp_ms'] for d in profile_results['baseline'])
st_total_attn = sum(d['attn_ms'] for d in profile_results['st_a2'])
st_total_mlp = sum(d['mlp_ms'] for d in profile_results['st_a2'])

x_pos = [0.25, 0.75]
bar_width = 0.3
bl_total = bl_total_attn + bl_total_mlp
st_total = st_total_attn + st_total_mlp

ax.barh(['ST-A²', 'Baseline'],
        [st_total_attn, bl_total_attn],
        color='#FF5722', alpha=0.85, label='Attention')
ax.barh(['ST-A²', 'Baseline'],
        [st_total_mlp, bl_total_mlp],
        left=[st_total_attn, bl_total_attn],
        color='#90CAF9', alpha=0.85, label='MLP/FFN')

ax.text(bl_total_attn/2, 1, f'{bl_total_attn:.0f}ms\n({bl_total_attn/bl_total*100:.0f}%)',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(bl_total_attn + bl_total_mlp/2, 1, f'{bl_total_mlp:.0f}ms\n({bl_total_mlp/bl_total*100:.0f}%)',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(st_total_attn/2, 0, f'{st_total_attn:.0f}ms\n({st_total_attn/st_total*100:.0f}%)',
        ha='center', va='center', fontsize=10, fontweight='bold')
ax.text(st_total_attn + st_total_mlp/2, 0, f'{st_total_mlp:.0f}ms\n({st_total_mlp/st_total*100:.0f}%)',
        ha='center', va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Total Encoder Forward Time (ms)')
ax.set_title('Total Time Breakdown: Attention vs MLP')
ax.legend(loc='lower right')

plt.suptitle('V-JEPA 2 ViT-L Per-Layer Profiling (256px, 16f, ~512 visible tokens)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('profiling_per_layer.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: profiling_per_layer.png')

In [ ]:
# Cell 10: Loss Curves

fig, ax = plt.subplots(1, 1, figsize=(10, 5))

# Smooth losses with rolling average
window = 10
for name, color, label in [('baseline', '#2196F3', 'Baseline (Full Attention)'),
                             ('st_a2', '#FF5722', 'ST-A\u00b2 (Area Attention)')]:
    raw = results[name]['losses']
    smoothed = pd.Series(raw).rolling(window=window, min_periods=1).mean()
    ax.plot(smoothed, color=color, linewidth=2, label=label)
    ax.plot(raw, color=color, alpha=0.15, linewidth=0.5)

ax.set_xlabel('Training Step', fontsize=12)
ax.set_ylabel('Loss (L1)', fontsize=12)
ax.set_title('V-JEPA 2 Loss Convergence: Baseline vs ST-A\u00b2', fontsize=14)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim(0, SHARED['num_steps'])
plt.tight_layout()
plt.savefig('ablation_loss_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_loss_curves.png')

In [ ]:
# Cell 11: Throughput & Memory Bar Charts

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
names = ['Baseline', 'ST-A\u00b2']
colors = ['#2196F3', '#FF5722']

# Step time
ax = axes[0]
vals = [bl['avg_step_ms'], st['avg_step_ms']]
bars = ax.bar(names, vals, color=colors, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f'{v:.0f}ms', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Time (ms)')
ax.set_title('Avg Step Time')
d = delta_pct(vals[0], vals[1])
ax.set_xlabel(f'({d:+.1f}%)', fontsize=11, color='green' if d < 0 else 'red')

# Peak memory
ax = axes[1]
vals = [bl['peak_mem_mb'], st['peak_mem_mb']]
bars = ax.bar(names, vals, color=colors, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{v:.0f}MB', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Memory (MB)')
ax.set_title('Peak GPU Memory')
d = delta_pct(vals[0], vals[1])
ax.set_xlabel(f'({d:+.1f}%)', fontsize=11, color='green' if d < 0 else 'red')

# Throughput
ax = axes[2]
vals = [bl['throughput_steps_sec'], st['throughput_steps_sec']]
bars = ax.bar(names, vals, color=colors, width=0.5)
for bar, v in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{v:.2f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Steps/sec')
ax.set_title('Throughput')
d = delta_pct(vals[0], vals[1])
ax.set_xlabel(f'({d:+.1f}%)', fontsize=11, color='green' if d > 0 else 'red')

plt.suptitle('ST-A\u00b2 Ablation: Performance Comparison', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('ablation_bar_charts.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ablation_bar_charts.png')

In [ ]:
# Cell 12: Save Results to CSV

H = W = SHARED['crop_size'] // SHARED['patch_size']
T = SHARED['num_frames'] // SHARED['tubelet_size']
total_tokens = H * W * T

# Per-step metrics
rows = []
for name in results:
    r = results[name]
    for i in range(len(r['losses'])):
        rows.append({
            'config': name,
            'step': i + 1,
            'loss': r['losses'][i],
            'step_time_ms': r['step_times_ms'][i],
        })
df_steps = pd.DataFrame(rows)
df_steps.to_csv('ablation_results.csv', index=False)
print(f'Saved per-step metrics: ablation_results.csv ({len(df_steps)} rows)')

# Summary with config columns
summary_rows = []
for name in results:
    r = results[name]
    cfg = CONFIGS[name]
    summary_rows.append({
        'config': name,
        'model': cfg['model_name'],
        'crop_size': cfg['crop_size'],
        'num_frames': cfg['num_frames'],
        'batch_size': cfg['batch_size'],
        'total_tokens': total_tokens,
        'visible_tokens_approx': total_tokens // 4,
        'use_area_attention': cfg['use_area_attention'],
        'area_layers': str(cfg.get('area_attention_layers', 'N/A')),
        'num_areas': cfg.get('area_spatial_splits', 1) * cfg.get('area_temporal_splits', 1) if cfg['use_area_attention'] else 1,
        'num_steps': cfg['num_steps'],
        'dtype': str(DTYPE),
        'gpu': gpu_name,
        'final_loss': r['final_loss'],
        'avg_step_ms': r['avg_step_ms'],
        'peak_mem_mb': r['peak_mem_mb'],
        'throughput_steps_sec': r['throughput_steps_sec'],
    })
df_summary = pd.DataFrame(summary_rows)
df_summary.to_csv('ablation_summary.csv', index=False)
print(f'Saved summary: ablation_summary.csv')
print()
print(df_summary.to_string(index=False))
print()
print('Done! Download ablation_results.csv and ablation_summary.csv for further analysis.')